# WavLM Evaluation - Centroid Analysis & Visualization

This notebook provides a comprehensive evaluation of WavLM embeddings using hierarchical centroid analysis.
It includes:
1. **Speaker Assignment**: Evaluating acoustic uniqueness.
2. **Distance Tables**: Statistical breakdown of distances to centroids.
3. **Interactive Visualization**: Spider plots and global centroid maps.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Display settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Add root directory to sys.path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from embeddings_eval.data_loader import load_embeddings, load_metadata
from embeddings_eval.analyzer import WavLMAnalyzer
from embeddings_eval.reporter import WavLMReporter
from embeddings_eval.visualizer import WavLMVisualizer
from embeddings_eval.constants import ALL_GROUPS

In [ ]:
DATA_DIR = "../datalocal/PC-GITA_v260210_24kHz/speaker_embeddings/wavLM"
META_PATH = "../datalocal/PC-GITA_v260210_24kHz/_metadata/PCGITAtoPD_mapping.csv"

print("Loading data and metadata...")
metadata = load_metadata(META_PATH)
embeddings = load_embeddings(DATA_DIR, metadata=metadata)
analyzer = WavLMAnalyzer(embeddings)
reporter = WavLMReporter(analyzer)
visualizer = WavLMVisualizer(analyzer)

print(f"Loaded {len(embeddings)} samples across {len(analyzer.speaker_centroids[analyzer.versions[0]])} speakers.")

# 1. Centroid-Based Speaker Assignment

Evaluates how successfully individual files are assigned to the correct speaker centroid.

In [ ]:
class_df = reporter.generate_classification_report()
agg_df_centroid = reporter.generate_aggregated_classification_report(class_df)

total_all = agg_df_centroid['total samples'].sum()
correct_all = agg_df_centroid['correct samples'].sum()
accuracy_all = (correct_all / total_all) * 100

print("--- GLOBAL CENTROID SUMMARY ---")
print(f"Total samples: {total_all}")
print(f"Correctly assigned to speaker: {correct_all}")
print(f"Global Accuracy: {accuracy_all:.2f}%")
print("------------------------------")

display(agg_df_centroid)
print("\nDetailed Results (Top 5 Intruders):")
display(class_df)

# 2. Statistical Distance Tables

Detailed Mean/Variance metrics for distances to group and speaker centroids.

In [ ]:
detailed_df = reporter.generate_detailed_report(calculate_samples=False)
summaries = reporter.generate_summary_reports(detailed_df)

print("PD Group Summary:")
display(summaries['PD'])
print("\nHC Group Summary:")
display(summaries['HC'])

In [ ]:
print("Detailed Speaker Search:")
def show_speaker_details(speaker_id):
    filtered = detailed_df[detailed_df['speaker id'] == speaker_id]
    display(filtered)

speaker_list = sorted(detailed_df['speaker id'].unique())
widgets.interact(show_speaker_details, speaker_id=speaker_list);

# 3. Visual Exploration

## 3.1 Global Centroid Map
Overall clusters of all speakers and their task-group centroids.

In [ ]:
visualizer.plot_centroids(method='pca')

## 3.2 Interactive Spider Plots
Select speakers to see hierarchical connections: Sample -> Group Centroid -> Speaker Centroid.

In [ ]:
def update_plot(method, selected_speakers):
    if not selected_speakers: return
    visualizer.plot_speaker_comparison(list(selected_speakers), method=method.lower())

ver = analyzer.versions[0]
speakers = sorted(list(analyzer.speaker_centroids[ver].keys()))
method_dropdown = widgets.Dropdown(options=['PCA', 't-SNE'], value='PCA', description='Method:')
speaker_select = widgets.SelectMultiple(options=speakers, value=[speakers[0]], description='Speakers:', rows=10)

widgets.interactive(update_plot, method=method_dropdown, selected_speakers=speaker_select)